# 07. Финальная проверка на test

Все решения – модели, параметры, признаки, правила ленты – принимались на val (173 пользователя). Из-за этого числа на val немного оптимистичны: мы выбирали то, что лучше сработало именно на этих людях. Test – ещё 173 пользователя, на которых не решалось ничего; их оценка показывает, как выбранный пайплайн работает на новых людях.

Правило: test смотрим **один раз**, после того как пайплайн зафиксирован. Если после этого что-то подкрутить и проверить снова, test станет вторым val.

Код: `packages/training/src/training/pipelines/final_test.py`. Стадии меряются на одних и тех же пользователях; `diff_to_previous` – парная разница `ndcg@10` с предыдущей строкой и её 95% интервал.

In [1]:
import pandas as pd

from training.pipelines import final_test
from training.reranking import evaluate as rerank_eval

results = final_test.evaluate_stages()
context = rerank_eval.load_context("test")
criterion = rerank_eval.criterion_thresholds(context)
print(f"пользователей: {len(context.cases)}; критерий: ndcg@10 >= {criterion.ndcg_min:.3f}, coverage@10 >= {criterion.coverage_min:.3f}")
test_table = final_test.stage_table(results, criterion)
test_table.round(4)

D:\Projects\Python\ml\shikimori-rec-sys\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


пользователей: 173; критерий: ndcg@10 >= 0.202, coverage@10 >= 0.123


,ndcg@10,ndcg_low,ndcg_high,diff_to_previous,diff_low,diff_high,hit_rate@10,coverage@10,franchise_repeats,max_franchise,no_prequel,критерий,recall@100
популярность,0.2249,0.1913,0.2624,0.0000,0.0000,0.0000,0.7168,0.0123,0.1879,2.6821,0.2775,False,0.3029
iALS,0.5221,0.4812,0.5619,0.2973,0.2616,0.3335,0.9538,0.0926,0.1104,1.8266,0.1145,False,0.5836
CatBoost,0.6132,0.5720,0.6546,0.0911,0.0653,0.1153,0.9653,0.0847,0.0931,1.8035,0.0295,False,NaN
CatBoost + re-ranking,0.5495,0.5093,0.5895,-0.0637,-0.0820,-0.0449,0.9538,0.1249,0.0000,1.0000,0.0000,True,NaN


Те же стадии на val (из ноутбуков 04–06) рядом с test:

In [2]:
val_ndcg = pd.Series({"популярность": 0.229, "iALS": 0.484, "CatBoost": 0.590, "CatBoost + re-ranking": 0.516})
val_coverage = pd.Series({"популярность": 0.012, "iALS": 0.085, "CatBoost": 0.081, "CatBoost + re-ranking": 0.121})
pd.DataFrame({
    "ndcg@10 val": val_ndcg,
    "ndcg@10 test": test_table["ndcg@10"],
    "coverage@10 val": val_coverage,
    "coverage@10 test": test_table["coverage@10"],
}).round(3)

,ndcg@10 val,ndcg@10 test,coverage@10 val,coverage@10 test
популярность,0.229,0.225,0.012,0.012
iALS,0.484,0.522,0.085,0.093
CatBoost,0.590,0.613,0.081,0.085
CatBoost + re-ranking,0.516,0.550,0.121,0.125


Вывод:

- **Результаты на test повторяют val.** Каждая стадия даёт то же, что на val: iALS сильно лучше популярности (`ndcg@10` 0,522 против 0,225, +0,297 [0,262; 0,334]), CatBoost добавляет к iALS +0,091 [0,065; 0,115] (на val +0,106), re-ranking забирает −0,064 [−0,082; −0,045] точности ради разнообразия (на val −0,074). Подбор на val не «переобучил» пайплайн под тех 173 человек.
- **Retrieval:** `recall@100` у iALS 0,584 (на val 0,582) – ranking получает в 100 кандидатах около 58% спрятанных ответов; у популярности 0,303.
- **Итоговая лента выполняет критерий и на test:** `ndcg@10` 0,550 при пороге 0,202, `coverage@10` 0,125 при пороге 0,123. Запас по покрытию маленький, как и на val: штраф 0,7 подбирался как наименьший, проходящий порог.
- **Правила ленты работают:** повторов франшиз и продолжений без приквела в итоговой ленте нет. У чистого CatBoost продолжений без приквела 3% мест, у iALS 11%, у популярности 28% (она показывает всем вторые сезоны хитов).
- **Чего test не показывает.** Это та же офлайн-постановка: «досмотрит ли человек спрятанное аниме», без показов и кликов. Эффект ленты на поведение (что человек выберет из показанного, надоест ли ему) проверяется только симуляцией обратной связи или A/B-тестом. Симуляция на этих же 173 пользователях – в `08_simulation.ipynb`: за 10 показов система даёт 11,8 досмотра на пользователя против 6,2 у популярной ленты.